Get the Dataset from here: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce?resource=download

My Directory Structure: 

##### aida-g2-retail-data-platform

##### ├── olist_dataset/
##### └── notebooks/DataPrep&Cleaning.ipynb

In [1]:
from pathlib import Path

folder_path = Path("../olist_dataset")

# Get and sort all CSV filenames in the folder
csv_files = sorted([file.name for file in folder_path.glob("*.csv")])

for idx, file_name in enumerate(csv_files, start=1):
    print(f"{idx}. {file_name}")

1. olist_customers_dataset.csv
2. olist_geolocation_dataset.csv
3. olist_order_items_dataset.csv
4. olist_order_payments_dataset.csv
5. olist_order_reviews_dataset.csv
6. olist_orders_dataset.csv
7. olist_products_dataset.csv
8. olist_sellers_dataset.csv
9. product_category_name_translation.csv


In [3]:
import pandas as pd
from pathlib import Path

def validate_olist_dataset(data_dir: str = "../olist_dataset"):
    path = Path(data_dir)
    print("Loading datasets for validation...\n")
    
    # Load all tables
    orders = pd.read_csv(path / "olist_orders_dataset.csv")
    items = pd.read_csv(path / "olist_order_items_dataset.csv")
    payments = pd.read_csv(path / "olist_order_payments_dataset.csv")
    reviews = pd.read_csv(path / "olist_order_reviews_dataset.csv")
    customers = pd.read_csv(path / "olist_customers_dataset.csv")
    products = pd.read_csv(path / "olist_products_dataset.csv")
    sellers = pd.read_csv(path / "olist_sellers_dataset.csv")
    translations = pd.read_csv(path / "product_category_name_translation.csv")
    geo = pd.read_csv(path / "olist_geolocation_dataset.csv")

    results = []

    def log_check(category: str, check_name: str, passed: bool, details: str):
        status = "PASS" if passed else "WARN/FAIL"
        results.append({
            "Category": category,
            "Check": check_name,
            "Status": status,
            "Details": details
        })

    # 1. PRIMARY KEY UNIQUENESS & NULL CHECKS
    log_check(
        "Primary Keys", "Orders: unique & non-null order_id",
        orders["order_id"].notna().all() and orders["order_id"].is_unique,
        f"Total: {len(orders)}, Nulls: {orders['order_id'].isna().sum()}, Duplicates: {orders['order_id'].duplicated().sum()}"
    )
    log_check(
        "Primary Keys", "Customers: unique & non-null customer_id",
        customers["customer_id"].notna().all() and customers["customer_id"].is_unique,
        f"Total: {len(customers)}, Unique people (customer_unique_id): {customers['customer_unique_id'].nunique()}"
    )
    log_check(
        "Primary Keys", "Products: unique & non-null product_id",
        products["product_id"].notna().all() and products["product_id"].is_unique,
        f"Total: {len(products)}, Nulls: {products['product_id'].isna().sum()}"
    )
    log_check(
        "Primary Keys", "Sellers: unique & non-null seller_id",
        sellers["seller_id"].notna().all() and sellers["seller_id"].is_unique,
        f"Total: {len(sellers)}, Nulls: {sellers['seller_id'].isna().sum()}"
    )
    log_check(
        "Primary Keys", "Reviews: unique review_id",
        reviews["review_id"].is_unique,
        f"Total rows: {len(reviews)}, Duplicate review_ids: {reviews['review_id'].duplicated().sum()}"
    )

    # 2. REFERENTIAL INTEGRITY (FOREIGN KEYS)
    missing_cust = set(orders["customer_id"]) - set(customers["customer_id"])
    log_check(
        "Foreign Keys", "Orders -> Customers (customer_id)",
        len(missing_cust) == 0,
        f"Orphaned customer_ids in orders: {len(missing_cust)}"
    )

    orders_without_items = set(orders["order_id"]) - set(items["order_id"])
    log_check(
        "Foreign Keys", "Orders -> Order Items (every order has >=1 item)",
        len(orders_without_items) == 0,
        f"Orders with 0 items: {len(orders_without_items)} (usually cancelled/unavailable orders)"
    )

    orders_without_payments = set(orders["order_id"]) - set(payments["order_id"])
    log_check(
        "Foreign Keys", "Orders -> Order Payments (every order has >=1 payment)",
        len(orders_without_payments) == 0,
        f"Orders with 0 payment records: {len(orders_without_payments)}"
    )

    missing_products = set(items["product_id"]) - set(products["product_id"])
    log_check(
        "Foreign Keys", "Order Items -> Products (product_id)",
        len(missing_products) == 0,
        f"Orphaned product_ids: {len(missing_products)}"
    )

    missing_sellers = set(items["seller_id"]) - set(sellers["seller_id"])
    log_check(
        "Foreign Keys", "Order Items -> Sellers (seller_id)",
        len(missing_sellers) == 0,
        f"Orphaned seller_ids: {len(missing_sellers)}"
    )

    untranslated_cats = set(products["product_category_name"].dropna()) - set(translations["product_category_name"])
    log_check(
        "Foreign Keys", "Products -> Category Translations",
        len(untranslated_cats) == 0,
        f"Categories missing English translation: {untranslated_cats if untranslated_cats else 0}"
    )

    # 3. DOMAIN & VALUE LOGIC CHECKS
    neg_prices = (items["price"] <= 0).sum()
    neg_freight = (items["freight_value"] < 0).sum()
    log_check(
        "Business Logic", "Item prices > 0 & freight >= 0",
        neg_prices == 0 and neg_freight == 0,
        f"Price <= 0: {neg_prices}, Freight < 0: {neg_freight}"
    )

    zero_payments = (payments["payment_value"] <= 0).sum()
    log_check(
        "Business Logic", "Payment values > 0",
        zero_payments == 0,
        f"Payments <= 0: {zero_payments} (vouchers/zero-value edge cases)"
    )

    invalid_scores = (~reviews["review_score"].isin([1, 2, 3, 4, 5])).sum()
    log_check(
        "Business Logic", "Review scores strictly between 1 and 5",
        invalid_scores == 0,
        f"Invalid review scores: {invalid_scores}"
    )

    # 4. TEMPORAL / CHRONOLOGICAL CONSISTENCY
    ts_cols = [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date"
    ]
    for col in ts_cols:
        orders[col] = pd.to_datetime(orders[col])

    # Purchase should happen before or at approval
    bad_approval = (orders["order_approved_at"] < orders["order_purchase_timestamp"]).sum()
    # Carrier handoff should happen before or at customer delivery
    bad_delivery = (orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"]).sum()
    log_check(
        "Timestamps", "Chronological order lifecycle (Purchase <= Approved, Carrier <= Delivered)",
        bad_approval == 0 and bad_delivery == 0,
        f"Approved before purchase: {bad_approval} | Delivered before carrier handoff: {bad_delivery}"
    )

    # 5. FINANCIAL RECONCILIATION (Items + Freight vs. Total Payment)
    item_totals = items.groupby("order_id").apply(lambda df: (df["price"] + df["freight_value"]).sum()).rename("expected_total")
    pay_totals = payments.groupby("order_id")["payment_value"].sum().rename("paid_total")
    recon = pd.concat([item_totals, pay_totals], axis=1).dropna()
    mismatches = (recon["expected_total"] - recon["paid_total"]).abs() > 0.05  # 5-cent rounding tolerance
    log_check(
        "Financials", "Order total (price + freight) matches sum of payments",
        mismatches.sum() == 0,
        f"Orders with >$0.05 payment discrepancy: {mismatches.sum()} out of {len(recon)}"
    )

    # 6. GEOLOCATION DUPLICATE CHECK
    dup_zips = geo["geolocation_zip_code_prefix"].duplicated().sum()
    log_check(
        "Dimensions", "Geolocation zip code uniqueness",
        dup_zips == 0,
        f"Total rows: {len(geo)}, Duplicate zip prefixes: {dup_zips} (must aggregate before joining!)"
    )

    # Display Report
    report_df = pd.DataFrame(results)
    print(report_df.to_string(index=False))
    return report_df

if __name__ == "__main__":
    validate_olist_dataset("../olist_dataset")

Loading datasets for validation...



C:\Users\jaivi\AppData\Local\Temp\ipykernel_26836\833117312.py:144: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  item_totals = items.groupby("order_id").apply(lambda df: (df["price"] + df["freight_value"]).sum()).rename("expected_total")


      Category                                                                      Check    Status                                                                                               Details
  Primary Keys                                         Orders: unique & non-null order_id      PASS                                                                 Total: 99441, Nulls: 0, Duplicates: 0
  Primary Keys                                   Customers: unique & non-null customer_id      PASS                                               Total: 99441, Unique people (customer_unique_id): 96096
  Primary Keys                                     Products: unique & non-null product_id      PASS                                                                                Total: 32951, Nulls: 0
  Primary Keys                                       Sellers: unique & non-null seller_id      PASS                                                                                 Total: 3095,

In [4]:
import pandas as pd
import json
from pathlib import Path
from datetime import datetime, timedelta

class OlistAPISimulator:
    def __init__(self, data_dir: str = "../olist_dataset"):
        self.data_dir = Path(data_dir)
        self._load_datasets()

    def _load_datasets(self):
        """Loads raw CSVs into memory to act as the mock backend database."""
        self.orders = pd.read_csv(
            self.data_dir / "olist_orders_dataset.csv",
            parse_dates=["order_purchase_timestamp"]
        )
        self.items = pd.read_csv(self.data_dir / "olist_order_items_dataset.csv")
        self.payments = pd.read_csv(self.data_dir / "olist_order_payments_dataset.csv")
        self.reviews = pd.read_csv(self.data_dir / "olist_order_reviews_dataset.csv")
        self.customers = pd.read_csv(self.data_dir / "olist_customers_dataset.csv")
        self.products = pd.read_csv(self.data_dir / "olist_products_dataset.csv")
        self.sellers = pd.read_csv(self.data_dir / "olist_sellers_dataset.csv")
        self.translations = pd.read_csv(self.data_dir / "product_category_name_translation.csv")
        self.geolocation = pd.read_csv(self.data_dir / "olist_geolocation_dataset.csv")

    def get_orders_batch(self, target_date: str, window_days: int = 1) -> list[dict]:
        """
        Simulates GET /api/v1/orders?start_date=YYYY-MM-DD&days=1
        Returns nested JSON records just like a real e-commerce REST API.
        """
        start_dt = pd.to_datetime(target_date)
        end_dt = start_dt + timedelta(days=window_days)

        # Filter orders within the requested time window
        batch_orders = self.orders[
            (self.orders["order_purchase_timestamp"] >= start_dt) &
            (self.orders["order_purchase_timestamp"] < end_dt)
        ].copy()

        if batch_orders.empty:
            return []

        order_ids = batch_orders["order_id"].tolist()

        # Pre-filter child tables for fast lookup
        batch_items = self.items[self.items["order_id"].isin(order_ids)]
        batch_payments = self.payments[self.payments["order_id"].isin(order_ids)]
        batch_reviews = self.reviews[self.reviews["order_id"].isin(order_ids)]
        batch_customers = self.customers[self.customers["customer_id"].isin(batch_orders["customer_id"])]

        # Build nested API response payload
        api_response = []
        for _, order in batch_orders.iterrows():
            oid = order["order_id"]
            cid = order["customer_id"]

            cust_record = batch_customers[batch_customers["customer_id"] == cid].to_dict(orient="records")
            
            payload = {
                "order_id": oid,
                "order_status": order["order_status"],
                "timestamps": {
                    "purchased_at": str(order["order_purchase_timestamp"]),
                    "approved_at": str(order["order_approved_at"]),
                    "delivered_carrier_at": str(order["order_delivered_carrier_date"]),
                    "delivered_customer_at": str(order["order_delivered_customer_date"]),
                    "estimated_delivery_at": str(order["order_estimated_delivery_date"]),
                },
                "customer": cust_record[0] if cust_record else None,
                "items": batch_items[batch_items["order_id"] == oid].drop(columns=["order_id"]).to_dict(orient="records"),
                "payments": batch_payments[batch_payments["order_id"] == oid].drop(columns=["order_id"]).to_dict(orient="records"),
                "reviews": batch_reviews[batch_reviews["order_id"] == oid].drop(columns=["order_id"]).to_dict(orient="records"),
            }
            api_response.append(payload)

        return api_response

    def get_catalog_reference(self) -> dict:
        """Simulates GET /api/v1/catalog (dimension lookup tables)."""
        products_enriched = self.products.merge(
            self.translations, on="product_category_name", how="left"
        )
        return {
            "products": products_enriched.to_dict(orient="records"),
            "sellers": self.sellers.to_dict(orient="records"),
        }


# --- Example Usage: Simulating a Daily Ingestion Run ---
if __name__ == "__main__":
    api = OlistAPISimulator(data_dir="../olist_dataset")

    # Simulate scraping orders for November 24, 2017 (Black Friday spike in Olist!)
    simulated_date = "2017-11-24"
    orders_payload = api.get_orders_batch(target_date=simulated_date, window_days=1)

    print(f"Fetched {len(orders_payload)} orders for {simulated_date}")
    if orders_payload:
        print("\nSample JSON Payload (First Order):")
        print(json.dumps(orders_payload[0], indent=2, default=str))

Fetched 1176 orders for 2017-11-24

Sample JSON Payload (First Order):
{
  "order_id": "6ea2f835b4556291ffdc53fa0b3b95e8",
  "order_status": "delivered",
  "timestamps": {
    "purchased_at": "2017-11-24 21:27:48",
    "approved_at": "2017-11-25 00:21:09",
    "delivered_carrier_at": "2017-12-13 21:14:05",
    "delivered_customer_at": "2017-12-28 18:59:23",
    "estimated_delivery_at": "2017-12-21 00:00:00"
  },
  "customer": {
    "customer_id": "c7340080e394356141681bd4c9b8fe31",
    "customer_unique_id": "3e4fd73f1e86b135b9b121d6abbe9597",
    "customer_zip_code_prefix": 19400,
    "customer_city": "presidente venceslau",
    "customer_state": "SP"
  },
  "items": [
    {
      "order_item_id": 1,
      "product_id": "be021417a6acb56b9b50d3fd2714baa8",
      "seller_id": "f5f46307a4d15880ca14fab4ad9dfc9b",
      "shipping_limit_date": "2017-11-30 00:21:09",
      "price": 339.0,
      "freight_value": 17.12
    }
  ],
  "payments": [
    {
      "payment_sequential": 1,
      "payme

Test Run

In [4]:
pip install -r ../ingestion/requirements.txt

     ---------------------------------------- 0.0/44.3 kB ? eta -:--:--
     ----------------- -------------------- 20.5/44.3 kB 682.7 kB/s eta 0:00:01
     -------------------------------------- 44.3/44.3 kB 549.7 kB/s eta 0:00:00
  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
     ---------------------------------------- 0.0/44.3 kB ? eta -:--:--
     ---------------------------------------- 44.3/44.3 kB 2.1 MB/s eta 0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
     ---------------------------------------- 0.0/40.8 kB ? eta -:--

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gtts 2.5.1 requires click<8.2,>=7.1, but you have click 8.5.1 which is incompatible.
tensorflow-intel 2.16.1 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.20.3, but you have protobuf 6.33.6 which is incompatible.
tensorflow-metadata 1.15.0 requires protobuf<4.21,>=3.20.3; python_version < "3.11", but you have protobuf 6.33.6 which is incompatible.


In [9]:
import os
import io
import sys
import json
import time
import logging
from pathlib import Path
from datetime import datetime, timedelta
from dotenv import load_dotenv, find_dotenv
from databricks.sdk import WorkspaceClient
from databricks import sql

In [10]:
# 1. Add project root to sys.path so we can import ingestion.simulator cleanly
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from ingestion.simulator import OlistAPISimulator

In [11]:
# 2. Load .env from project root and validate required variables
env_path = find_dotenv(usecwd=True) or str(project_root / ".env")
load_dotenv(env_path, override=True)

required_vars = ["DATABRICKS_HOST", "DATABRICKS_HTTP_PATH", "DATABRICKS_TOKEN"]
missing = [v for v in required_vars if not os.getenv(v)]
if missing:
    raise EnvironmentError(f"Missing required environment variables in {env_path}: {missing}")

HOST = os.environ["DATABRICKS_HOST"].strip().rstrip("/")
HTTP_PATH = os.environ["DATABRICKS_HTTP_PATH"].strip()
TOKEN = os.environ["DATABRICKS_TOKEN"].strip()

In [12]:
# 3. Configure immediate unbuffered notebook logging
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    stream=sys.stdout,
    force=True
)
logger = logging.getLogger("olist_test_runner")

CATALOG = "olist_dev"
SCHEMA = "bronze"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/raw_landing"


def test_databricks_ingestion(test_start="2017-11-24", test_end="2017-11-24"):
    t0 = time.time()
    logger.info(f"Step 1/5: Initializing OlistAPISimulator from {project_root / 'olist_dataset'}...")
    sim = OlistAPISimulator(data_dir=project_root / "olist_dataset")

    start_dt = datetime.strptime(test_start, "%Y-%m-%d").date()
    end_dt = datetime.strptime(test_end, "%Y-%m-%d").date()
    window_days = (end_dt - start_dt).days + 1

    records = sim.get_orders_batch(target_date=test_start, window_days=window_days)
    logger.info(f"Step 2/5: Generated {len(records)} nested JSON records for {test_start} ({time.time() - t0:.1f}s)")

    if not records:
        logger.warning("No records found for test window.")
        return

    # Upload as NDJSON (1 JSON object per line) to Unity Catalog Volume
    logger.info("Step 3/5: Uploading NDJSON batch to Databricks Unity Catalog Volume...")
    w = WorkspaceClient(host=HOST, token=TOKEN)
    file_name = f"orders_batch_{test_start}_to_{test_end}.jsonl"
    target_volume_file = f"{VOLUME_PATH}/{file_name}"

    ndjson_bytes = "\n".join(json.dumps(rec, default=str) for rec in records).encode("utf-8")
    w.files.upload(target_volume_file, io.BytesIO(ndjson_bytes), overwrite=True)
    logger.info(f"Upload complete -> {target_volume_file} ({time.time() - t0:.1f}s)")

    # Connect to SQL Warehouse (note: cold start on Free Edition warehouse takes ~10-20s)
    logger.info("Step 4/5: Connecting to Databricks SQL Warehouse (waking up serverless compute if idle)...")
    with sql.connect(
        server_hostname=HOST.replace("https://", ""),
        http_path=HTTP_PATH,
        access_token=TOKEN,
        _socket_timeout=120
    ) as conn:
        with conn.cursor() as cursor:
            logger.info("Connected! Ensuring Bronze tables exist...")
            cursor.execute(f"""
                CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.ingestion_watermark (
                    pipeline_name STRING,
                    last_processed_date DATE,
                    records_loaded INT,
                    updated_at TIMESTAMP
                ) USING DELTA
            """)

            cursor.execute(f"""
                CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.orders_raw_json (
                    order_id STRING,
                    batch_date DATE,
                    ingested_at TIMESTAMP,
                    raw_payload STRING
                ) USING DELTA
            """)

            logger.info("Step 5/5: Merging NDJSON from Volume into Bronze Delta table...")
            cursor.execute(f"""
                MERGE INTO {CATALOG}.{SCHEMA}.orders_raw_json AS target
                USING (
                    SELECT
                        order_id,
                        CAST('{test_end}' AS DATE) AS batch_date,
                        CURRENT_TIMESTAMP() AS ingested_at,
                        to_json(struct(*)) AS raw_payload
                    FROM read_files('{target_volume_file}', format => 'json')
                ) AS source
                ON target.order_id = source.order_id
                WHEN NOT MATCHED THEN INSERT *
            """)

            cursor.execute(f"""
                INSERT INTO {CATALOG}.{SCHEMA}.ingestion_watermark
                VALUES ('olist_orders_api', CAST('{test_end}' AS DATE), {len(records)}, CURRENT_TIMESTAMP())
            """)

            # Verify row count directly from Databricks
            cursor.execute(f"SELECT COUNT(*) FROM {CATALOG}.{SCHEMA}.orders_raw_json")
            total_rows = cursor.fetchone()[0]
            logger.info(f"SUCCESS! Total rows in {CATALOG}.{SCHEMA}.orders_raw_json: {total_rows} (Total time: {time.time() - t0:.1f}s)")

In [13]:
# Run the smoke test
test_databricks_ingestion(test_start="2017-11-24", test_end="2017-11-24")

2026-11-19 13:29:31,761 | INFO | Step 1/5: Initializing OlistAPISimulator from C:\Users\jaivi\Videos\AIDA\Projects\aida-g2-retail-data-platform\olist_dataset...
2026-11-19 13:33:15,276 | INFO | Step 2/5: Generated 1176 nested JSON records for 2017-11-24 (223.5s)
2026-11-19 13:33:15,277 | INFO | Step 3/5: Uploading NDJSON batch to Databricks Unity Catalog Volume...
2026-11-19 13:33:16,169 | INFO | Upload complete -> /Volumes/olist_dev/bronze/raw_landing/orders_batch_2017-11-24_to_2017-11-24.jsonl (224.4s)
2026-11-19 13:33:16,170 | INFO | Step 4/5: Connecting to Databricks SQL Warehouse (waking up serverless compute if idle)...
2026-11-19 13:33:17,192 | INFO | Received status code 200 for POST request
2026-11-19 13:33:17,192 | INFO | HTTP Response with status code 200, message: OK
2026-11-19 13:33:17,194 | INFO | Successfully opened session 01f1e447-b00e-1055-9a68-f7e569a4bf3b
2026-11-19 13:33:17,448 | INFO | Connected! Ensuring Bronze tables exist...
2026-11-19 13:33:20,287 | INFO | Rec

In [8]:
import os
import time
import requests
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from databricks import sql

# 1. Load .env from project root
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
env_path = find_dotenv(usecwd=True) or str(project_root / ".env")
load_dotenv(env_path, override=True)

HOST = os.environ["DATABRICKS_HOST"].strip().rstrip("/")
HTTP_PATH = os.environ["DATABRICKS_HTTP_PATH"].strip()
TOKEN = os.environ["DATABRICKS_TOKEN"].strip()

print(f"Testing connection to: {HOST}\n")

# --- TEST 1: REST API & Token Authentication ---
t0 = time.time()
try:
    resp = requests.get(
        f"{HOST}/api/2.0/preview/scim/v2/Me",
        headers={"Authorization": f"Bearer {TOKEN}"},
        timeout=15
    )
    resp.raise_for_status()
    user_email = resp.json().get("userName", "Authenticated User")
    print(f"[PASS] 1. Token & REST API valid ({time.time() - t0:.1f}s) -> Logged in as: {user_email}")
except Exception as e:
    print(f"[FAIL] 1. REST API / Token check failed: {e}")

# --- TEST 2: Unity Catalog Volume Access ---
t1 = time.time()
volume_dir = "/Volumes/olist_dev/bronze/raw_landing"
try:
    resp = requests.get(
        f"{HOST}/api/2.0/fs/directories{volume_dir}",
        headers={"Authorization": f"Bearer {TOKEN}"},
        timeout=15
    )
    resp.raise_for_status()
    print(f"[PASS] 2. Volume accessible ({time.time() - t1:.1f}s) -> {volume_dir}")
except Exception as e:
    print(f"[FAIL] 2. Volume check failed ({volume_dir}): {e}")

# --- TEST 3: Serverless SQL Warehouse Connection ---
t2 = time.time()
print("       3. Connecting to SQL Warehouse (may take 10-25s if cold-starting)...")
try:
    with sql.connect(
        server_hostname=HOST.replace("https://", ""),
        http_path=HTTP_PATH,
        access_token=TOKEN,
        _socket_timeout=60
    ) as conn:
        with conn.cursor() as cursor:
            cursor.execute("SELECT current_catalog(), current_schema(), current_timestamp()")
            row = cursor.fetchone()
            print(f"[PASS] 3. SQL Warehouse connected ({time.time() - t2:.1f}s) -> Result: {row}")
except Exception as e:
    print(f"[FAIL] 3. SQL Warehouse connection failed: {e}")

Testing connection to: https://dbc-a8ae7627-918a.cloud.databricks.com

[PASS] 1. Token & REST API valid (0.3s) -> Logged in as: jaivinjacobvj@gmail.com
[PASS] 2. Volume accessible (0.5s) -> /Volumes/olist_dev/bronze/raw_landing
       3. Connecting to SQL Warehouse (may take 10-25s if cold-starting)...
2026-11-19 13:29:10,755 | INFO | Received status code 200 for POST request
2026-11-19 13:29:10,756 | INFO | HTTP Response with status code 200, message: OK
2026-11-19 13:29:10,757 | INFO | Successfully opened session 01f1e447-1d29-113f-a931-329e52ed4544
2026-11-19 13:29:10,860 | INFO | Received status code 200 for GET request
2026-11-19 13:29:13,044 | INFO | Received status code 200 for POST request
2026-11-19 13:29:13,045 | INFO | HTTP Response with status code 200, message: OK
[PASS] 3. SQL Warehouse connected (3.5s) -> Result: Row(current_catalog()='workspace', current_schema()='default', current_timestamp()=datetime.datetime(2026, 11, 19, 18, 29, 12, 245162, tzinfo=<StaticTzInfo 'Etc

In [14]:
!python ../tests/bronze_loader_test.py

2026-11-19 15:34:04,083 | INFO | Step 1/5: Loading pre-indexed OlistAPISimulator...
2026-11-19 15:37:50,963 | INFO | Step 2/5: Generated 1176 nested JSON records (226.9s)
2026-11-19 15:37:50,963 | INFO | Step 3/5: Uploading NDJSON to /Volumes/olist_dev/bronze/raw_landing/orders_batch_2017-11-24_to_2017-11-24.jsonl...
2026-11-19 15:37:51,475 | INFO | Upload complete (227.4s)
2026-11-19 15:37:51,475 | INFO | Step 4/5: Connecting to Databricks SQL Warehouse...
2026-11-19 15:37:52,866 | INFO | Received status code 200 for POST request
2026-11-19 15:37:52,866 | INFO | HTTP Response with status code 200, message: OK
2026-11-19 15:37:52,867 | INFO | Successfully opened session 01f1e459-17f1-1451-a980-c855ffbc15ca
2026-11-19 15:37:52,986 | INFO | Received status code 200 for GET request
2026-11-19 15:37:53,909 | INFO | Received status code 200 for POST request
2026-11-19 15:37:53,909 | INFO | HTTP Response with status code 200, message: OK
2026-11-19 15:37:54,143 | INFO | Received status code 

In [15]:
!python ../ingestion/bronze_loader.py

2026-11-19 15:46:42,315 | INFO | Received status code 200 for POST request
2026-11-19 15:46:42,316 | INFO | HTTP Response with status code 200, message: OK
2026-11-19 15:46:42,316 | INFO | Successfully opened session 01f1e45a-5388-129e-9ad2-29e680482637
2026-11-19 15:46:42,776 | INFO | Received status code 200 for GET request
2026-11-19 15:46:43,514 | INFO | Received status code 200 for POST request
2026-11-19 15:46:43,515 | INFO | HTTP Response with status code 200, message: OK
2026-11-19 15:46:43,738 | INFO | Received status code 200 for POST request
2026-11-19 15:46:43,739 | INFO | HTTP Response with status code 200, message: OK
2026-11-19 15:46:44,331 | INFO | Received status code 200 for POST request
2026-11-19 15:46:44,331 | INFO | HTTP Response with status code 200, message: OK
2026-11-19 15:46:44,339 | INFO | No watermark found. Running INITIAL LOAD (2016-09-04 to 2017-12-31).
2026-11-19 15:46:59,327 | INFO | Uploaded 45430 records to /Volumes/olist_dev/bronze/raw_landing/order